# Phase 6 DAEAC Paper-Faithful Kaggle Driver

This notebook is a thin driver. It calls repository scripts instead of duplicating training logic.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, textwrap

# Edit this to your GitHub repo URL before running on Kaggle.
REPO_URL = 'https://github.com/YOUR_ORG/Best-thesis-in-council.git'
BRANCH = 'main'

WORK = Path('/kaggle/working')
REPO = WORK / 'Best-thesis-in-council'
ECG = REPO / 'ecg_thesis'
print('Kaggle working:', WORK.exists())
print('Repo path:', ECG)

## 1. Clone Repo From GitHub

Edit `REPO_URL` in the first cell, then run this cell. If the repo already exists in `/kaggle/working`, the cell reuses it.

In [ ]:
if not ECG.exists():
    assert 'YOUR_ORG' not in REPO_URL, 'Edit REPO_URL before cloning.'
    !git clone --branch {BRANCH} {REPO_URL} /kaggle/working/Best-thesis-in-council
else:
    print('Repo already exists:', ECG)
assert ECG.exists(), f'Missing repo at {ECG}. Clone or upload it first.'
os.chdir(ECG)
print(Path.cwd())
!git status --short || true

## 2. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Copy Preprocessed DAEAC Data

This notebook expects the Kaggle dataset structure shown by the user: a folder named `daeac` containing files such as `mitdb_ds1_daeac.npz`, `mitdb_ds2_daeac.npz`, and `mitdb_ds2_first5_unlabeled_daeac.npz`.

In [ ]:
# The dataset screenshot shows: /kaggle/input/<dataset_slug>/daeac/*.npz
candidate_dirs = [p for p in Path('/kaggle/input').glob('**/daeac') if p.is_dir()]
print('Candidate daeac data dirs:')
for p in candidate_dirs:
    print(' -', p)

KAGGLE_DATASET_DIR = candidate_dirs[0] if candidate_dirs else Path('/kaggle/input/YOUR_DATASET_NAME/daeac')
print('Using data dir:', KAGGLE_DATASET_DIR)

DEST = Path('data/processed/phase6_daeac_paper')
DEST.mkdir(parents=True, exist_ok=True)
print('Available npz files under /kaggle/input:')
for p in sorted(Path('/kaggle/input').glob('**/*.npz')):
    print(' -', p)

if KAGGLE_DATASET_DIR.exists():
    for p in KAGGLE_DATASET_DIR.glob('*.npz'):
        shutil.copy2(p, DEST / p.name)
        print('copied', p.name)
else:
    raise FileNotFoundError('Could not find the daeac dataset folder. Edit KAGGLE_DATASET_DIR manually.')

print('Copied files:')
!ls -lh data/processed/phase6_daeac_paper

## 4. Validate Repo and Data

In [ ]:
!python scripts/check_repo.py
!python scripts/phase6_daeac_paper/00_validate_data.py --config configs/phase6_daeac_paper.yaml

## 5. Smoke Run

In [ ]:
!python scripts/phase6_daeac_paper/01_train_base.py --config configs/phase6_daeac_paper.yaml --epochs 1 --max-source-samples 512 --max-val-samples 512
!python scripts/phase6_daeac_paper/02_adapt_uda.py --config configs/phase6_daeac_paper.yaml --epochs 1 --max-source-samples 512 --max-target-samples 512 --max-val-samples 512

## 6. Full Run

Run these cells after the smoke run passes. The default config uses 300 pretraining epochs and 300 adaptation epochs.

In [ ]:
# !python scripts/phase6_daeac_paper/01_train_base.py --config configs/phase6_daeac_paper.yaml
# !python scripts/phase6_daeac_paper/02_adapt_uda.py --config configs/phase6_daeac_paper.yaml

## 7. Evaluate and Report

In [ ]:
!python scripts/phase6_daeac_paper/03_eval.py --config configs/phase6_daeac_paper.yaml --checkpoint outputs/phase6_daeac_paper/checkpoints/daeac_base_latest.pt --method-name daeac_base --dataset target
!python scripts/phase6_daeac_paper/03_eval.py --config configs/phase6_daeac_paper.yaml --checkpoint outputs/phase6_daeac_paper/checkpoints/daeac_uda_latest.pt --method-name daeac_uda --dataset target
!python scripts/phase6_daeac_paper/04_make_report.py --config configs/phase6_daeac_paper.yaml

## 8. Zip Outputs

In [ ]:
!zip -r /kaggle/working/phase6_daeac_paper_outputs.zip outputs/phase6_daeac_paper